# Transformer building blocks

## 1. Positional encoding: how does attention learn the word order?

Notebook 03 ended with a gap: attention only compares vectors, so shuffling the tokens only shuffles the rows of the output.

- "A cute teddy bear" and "bear teddy cute A" give each token the same new vector.
- But word order matters: "the dog bit the man" is not "the man bit the dog".
- The fix: put the position into the embedding, before attention starts.

#### 1.1 The idea: add a position vector

![Each embedding plus its position vector gives a position-aware input](images/positional_add.svg)

- Every position (1st, 2nd, 3rd, ...) has its own **position vector** $p$, the same size as an embedding: $d_{\text{model}}$ numbers.
- The input to attention is $x + p$: the embedding plus the position vector, added element by element.
- The same word at a different position now gives a different input. "bear" as the 4th token and "bear" as the 1st token give two different vectors.
- Everything after this step stays as in notebook 03. Attention just receives $x + p$ instead of $x$.

#### 1.2 Where do the position vectors come from?

Two options (lecture: "can be either learned or hardcoded"):

- **Learned:** a table with one row per position, like the embedding table in notebook 01. The numbers start random and are trained with the rest of the model. The table has a fixed size, so there is a maximum length.
- **Fixed formula:** the original Transformer used sines and cosines. There is nothing to train.

$$p_{\text{pos},\,2i} = \sin\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right), \qquad p_{\text{pos},\,2i+1} = \cos\!\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

**How the formula fills $p$.** Take $d_{\text{model}} = 4$.

- $p$ has 4 slots, grouped in pairs. Each pair belongs to one **hand** and holds its $\sin$ and $\cos$. The letter $i$ counts the pairs:

| $i$ (hand) | Slots it fills | Numbers |
|---|---|---|
| 0 (fast hand) | slot 0 and slot 1 | $\sin$, $\cos$ |
| 1 (slow hand) | slot 2 and slot 3 | $\sin$, $\cos$ |

- Hand $i$ fills two neighbouring slots: slot $2i$ with its $\sin$ (first formula), slot $2i+1$ with its $\cos$ (second formula).

Plug $d_{\text{model}} = 4$ into the formulas, slot by slot:

- **Slot 0** has the form $2i$ with $i = 0$, so the first formula applies: $\sin\!\left(\dfrac{\text{pos}}{10000^{0/4}}\right) = \sin\!\left(\dfrac{\text{pos}}{1}\right)$.
- **Slot 1** has the form $2i+1$ with $i = 0$, so the second formula applies: $\cos\!\left(\dfrac{\text{pos}}{10000^{0/4}}\right) = \cos\!\left(\dfrac{\text{pos}}{1}\right)$.
- **Slot 2** has the form $2i$ with $i = 1$. The exponent is $2i / d_{\text{model}} = 2/4 = 0.5$, and $10000^{0.5} = 100$: $\sin\!\left(\dfrac{\text{pos}}{100}\right)$.
- **Slot 3** has the form $2i+1$ with $i = 1$, with the same divisor $100$: $\cos\!\left(\dfrac{\text{pos}}{100}\right)$.

Two things decide a slot: even or odd decides $\sin$ or $\cos$, and the hand number $i$ decides the divisor $10000^{2i/d_{\text{model}}}$ (the speed of the hand). Now the numbers for positions 0 and 1:

| Slot | Formula | Position 0 | Position 1 | Position 2 | Position 3 |
|---|---|---|---|---|---|
| 0 | $\sin(\text{pos} / 1)$ | $\sin(0) = 0.00$ | $\sin(1) = 0.84$ | $\sin(2) = 0.91$ | $\sin(3) = 0.14$ |
| 1 | $\cos(\text{pos} / 1)$ | $\cos(0) = 1.00$ | $\cos(1) = 0.54$ | $\cos(2) = -0.42$ | $\cos(3) = -0.99$ |
| 2 | $\sin(\text{pos} / 100)$ | $\sin(0) = 0.00$ | $\sin(0.01) = 0.01$ | $\sin(0.02) = 0.02$ | $\sin(0.03) = 0.03$ |
| 3 | $\cos(\text{pos} / 100)$ | $\cos(0) = 1.00$ | $\cos(0.01) = 1.00$ | $\cos(0.02) = 1.00$ | $\cos(0.03) = 1.00$ |

Reading each column from top to bottom gives the position vector:

- $p_0 = [0.00,\ 1.00,\ 0.00,\ 1.00]$
- $p_1 = [0.84,\ 0.54,\ 0.01,\ 1.00]$
- $p_2 = [0.91,\ -0.42,\ 0.02,\ 1.00]$
- $p_3 = [0.14,\ -0.99,\ 0.03,\ 1.00]$

Why sines and cosines? Each position needs its own vector of small numbers. A clock with several hands does this.

- **One hand per pair.** The position sets how far the hand has turned: angle = pos $\times$ speed. The tip of the hand is a point on a circle, with coordinates $(\cos, \sin)$. Both numbers are needed: two different angles can have the same $\sin$ (the same height), but then their $\cos$ differs.
- **Several speeds.** One hand repeats after a full turn (the fast hand every 6 positions or so). So we use hands of different speeds. A fast hand tells **neighbours** apart, a slow hand tells **far-away** positions apart, and together they give every position its own address.
- **A step is a turn.** Moving $k$ positions turns each hand by a fixed angle, the same at every position. So "3 words earlier" looks the same everywhere, which is easy for the model to learn.
- **Small values.** Every number is between $-1$ and $1$, so the position vector stays small next to the embedding.

![One position uses the fast and the slow hand together. Positions 0 and 44 look the same to the fast hand but not to the slow hand](images/positional_hands.svg)

#### 1.3 Positions make the order matter

We write the formula from 1.2 in PyTorch and check that it gives the numbers of the table.

In [1]:
import math
import torch

torch.set_printoptions(precision=2, sci_mode=False)


def positional_encoding(T, d_model):
    pos = torch.arange(T).unsqueeze(1)                     # 0, 1, 2, ... one row per position
    two_i = torch.arange(0, d_model, 2)                    # 0, 2, 4, ... the first slot of each hand
    angle = pos / 10000 ** (two_i / d_model)               # one angle per position and per hand
    P = torch.zeros(T, d_model)
    P[:, 0::2] = torch.sin(angle)                          # even slots: sin
    P[:, 1::2] = torch.cos(angle)                          # odd slots: cos
    return P


P = positional_encoding(T=4, d_model=4)
print("position vectors, one row per position:")
print(P)

position vectors, one row per position:
tensor([[ 0.00,  1.00,  0.00,  1.00],
        [ 0.84,  0.54,  0.01,  1.00],
        [ 0.91, -0.42,  0.02,  1.00],
        [ 0.14, -0.99,  0.03,  1.00]])


Now the shuffle test of notebook 03 (section 4.3), with and without positions. We shuffle the tokens, then give position vectors by place: the new first token gets position 0, and so on.

In [2]:
# Same numbers as notebook 03: one row per token (A, cute, teddy, bear)
X = torch.tensor([[ 0.1, 0.0,  0.3, -0.1],
                  [ 0.2, 0.5, -0.1,  0.4],
                  [ 0.7, 0.1, -0.3,  0.8],
                  [-0.1, 0.9,  0.4,  0.2]])
W_Q = torch.tensor([[0., 0.], [1., -1.], [1., -1.], [1., 0.]])
W_K = torch.tensor([[0., -1.], [1., 0.], [-1., 1.], [1., 1.]])
W_V = torch.tensor([[0., 1.], [1., 0.], [1., 0.], [0., 0.]])


def attention(X):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    weights = torch.softmax(Q @ K.T / math.sqrt(K.shape[-1]), dim=-1)
    return weights @ V


perm = [3, 1, 0, 2]                                        # new order: bear, cute, A, teddy

# Without positions (notebook 03): shuffling the input only shuffles the output rows
Z = attention(X)
Z_shuffled = attention(X[perm])
print("without positions, shuffle then attention == attention then shuffle:",
      torch.allclose(Z_shuffled, Z[perm], atol=1e-6))

# With positions: the shuffled tokens get position vectors by their new place
Z_pos = attention(X + P)
Z_pos_shuffled = attention(X[perm] + P)
print("with positions,    shuffle then attention == attention then shuffle:",
      torch.allclose(Z_pos_shuffled, Z_pos[perm], atol=1e-6))

print("\n'bear' as the 4th token:", Z_pos[3])
print("'bear' as the 1st token:", Z_pos_shuffled[0])

without positions, shuffle then attention == attention then shuffle: True
with positions,    shuffle then attention == attention then shuffle: False

'bear' as the 4th token: tensor([0.58, 0.98])
'bear' as the 1st token: tensor([1.09, 0.89])


- Without positions, the shuffled input gives the same vectors in a new order. Attention does not see the order.
- With positions, the check fails: "bear" gets a different output as the 1st token than as the 4th token.
- The position vector is part of the input, so the same word in a different place is a different input.

## 2. Multi-head attention

> A cute teddy bear is reading.

- In notebook 03, each token gets one table of weights. That is one way of looking at the other tokens.
- A token can have several things to look at. "bear" may want "cute" (what kind of bear) and "reading" (what it is doing). One set of weights has to split its attention between them.
- With several attentions in parallel, called **heads**, each one gets its own weights. Every head has its own $W_Q$, $W_K$ and $W_V$.
- It works like the filters of a convolution layer in vision: each filter picks up a different pattern.
- Each head learns for itself what to look at. "cute" and "reading" are just examples to help us picture it.

#### 2.1 The idea: several heads

![The same input goes through two heads, each with its own weights and its own output](images/multihead_two_heads.svg)

- Head 1 is the attention of notebook 03. For "bear" it listens mostly to "teddy" ($0.50$).
- Head 2 has its own $W_Q$, $W_K$ and $W_V$ (made up, like in notebook 03). It listens mostly to "cute" ($0.48$).
- The same input gives two different outputs, $[0.21,\ 0.40]$ and $[0.59,\ 0.46]$. Each one is a mix of that head's own values.
- Each head output has $d_v = 2$ numbers.

#### 2.2 Putting the heads together

$$\operatorname{MultiHead}(X) = \operatorname{Concat}(\text{head}_1, \dots, \text{head}_h)\, W_O, \qquad \text{head}_i = \operatorname{Attention}(X W_Q^{(i)},\ X W_K^{(i)},\ X W_V^{(i)})$$

- Each head runs the attention of notebook 03 on the same $X$, with its own $W_Q^{(i)}$, $W_K^{(i)}$ and $W_V^{(i)}$.
- **Concat** glues the head outputs side by side. $h$ heads with $d_v$ numbers each give $h \cdot d_v$ numbers per token.
- $W_O$ mixes the glued numbers, so each output number can use numbers from both heads. It gives each token $d_{\text{model}}$ numbers again, so the output of multi-head attention has the same shape as the input $X$.
- All these matrices ($W_Q^{(i)}$, $W_K^{(i)}$ and $W_V^{(i)}$ for every head, and $W_O$) are learned together with the rest of the model. In our examples they are made up.

![X goes through two heads of 4 x 2, Concat gives 4 x 4, W_O keeps 4 x 4, so the output has the shape of X](images/multihead_shapes.svg)

Shapes for our example: $T = 4$, $d_{\text{model}} = 4$, $h = 2$ heads, $d_k = d_v = 2$.

| Piece | Shape | Our example |
|---|---|---|
| $X$ | $T \times d_{\text{model}}$ | $4 \times 4$ |
| $W_Q^{(i)}$, $W_K^{(i)}$ | $d_{\text{model}} \times d_k$ | $4 \times 2$ |
| $W_V^{(i)}$ | $d_{\text{model}} \times d_v$ | $4 \times 2$ |
| $\text{head}_i$ | $T \times d_v$ | $4 \times 2$ |
| Concat | $T \times (h \cdot d_v)$ | $4 \times 4$ |
| $W_O$ | $(h \cdot d_v) \times d_{\text{model}}$ | $4 \times 4$ |
| $\operatorname{MultiHead}(X)$ | $T \times d_{\text{model}}$ | $4 \times 4$ |

- The original Transformer picks $d_k = d_v = d_{\text{model}} / h$ (here $4 / 2 = 2$). Then Concat has exactly $d_{\text{model}}$ numbers per token.

#### 2.3 Check in PyTorch

We run the two heads of 2.1, glue their outputs and multiply by $W_O$ (made up). Then we compare with the built-in `nn.MultiheadAttention`.

In [3]:
# One row per token (A, cute, teddy, bear), same numbers as before
X = torch.tensor([[ 0.1, 0.0,  0.3, -0.1],
                  [ 0.2, 0.5, -0.1,  0.4],
                  [ 0.7, 0.1, -0.3,  0.8],
                  [-0.1, 0.9,  0.4,  0.2]])

# Head 1 is the head of notebook 03, head 2 is the one of section 2.1. Each matrix is 4 x 2
W_Q = [torch.tensor([[0., 0.], [1., -1.], [1., -1.], [1., 0.]]),
       torch.tensor([[0., 0.], [-1., 2.], [0., 0.], [2., 2.]])]
W_K = [torch.tensor([[0., -1.], [1., 0.], [-1., 1.], [1., 1.]]),
       torch.tensor([[0., -2.], [-1., 0.], [2., -2.], [0., 0.]])]
W_V = [torch.tensor([[0., 1.], [1., 0.], [1., 0.], [0., 0.]]),
       torch.tensor([[1., 0.], [0., 1.], [0., 1.], [1., 0.]])]

# W_O is 4 x 4 (made up): it mixes the 4 glued numbers of each token
W_O = torch.tensor([[1., 0., 1., 0.],
                    [0., 1., 0., 1.],
                    [1., 0., 0., 1.],
                    [0., 1., 1., 0.]])


def attention(X, W_Q, W_K, W_V):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    weights = torch.softmax(Q @ K.T / math.sqrt(K.shape[-1]), dim=-1)
    return weights @ V


head_outputs = [attention(X, W_Q[i], W_K[i], W_V[i]) for i in range(2)]   # two 4 x 2 matrices
concat = torch.cat(head_outputs, dim=-1)                                  # side by side: 4 x 4
Z = concat @ W_O                                                          # mix: 4 x 4

print("X:               ", tuple(X.shape))
print("output of head 1:", tuple(head_outputs[0].shape))
print("output of head 2:", tuple(head_outputs[1].shape))
print("after Concat:    ", tuple(concat.shape))
print("after W_O:       ", tuple(Z.shape))

print("\n'bear' (last row)")
print("head 1:", head_outputs[0][3])
print("head 2:", head_outputs[1][3])
print("Concat:", concat[3])
print("W_O   :", Z[3])

X:                (4, 4)
output of head 1: (4, 2)
output of head 2: (4, 2)
after Concat:     (4, 4)
after W_O:        (4, 4)

'bear' (last row)
head 1: tensor([0.21, 0.40])
head 2: tensor([0.59, 0.46])
Concat: tensor([0.21, 0.40, 0.59, 0.46])
W_O   : tensor([0.80, 0.87, 0.67, 1.00])


In [4]:
# PyTorch's built-in multi-head attention, with the same 2 heads and the same matrices
mha = torch.nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)
with torch.no_grad():
    # it stores the W_Q of all heads side by side in one matrix (and the same for W_K and W_V)
    # and it multiplies as x @ weight.T, so we pass the transposes
    mha.in_proj_weight.copy_(torch.cat([torch.cat(W_Q, dim=1).T,
                                        torch.cat(W_K, dim=1).T,
                                        torch.cat(W_V, dim=1).T], dim=0))
    mha.out_proj.weight.copy_(W_O.T)
    mha.in_proj_bias.zero_()
    mha.out_proj.bias.zero_()

Z_torch, _ = mha(X[None], X[None], X[None])        # [None] adds a batch dimension

print("PyTorch output for 'bear':", Z_torch[0, 3].detach())
print("ours == PyTorch:", torch.allclose(Z, Z_torch[0].detach(), atol=1e-5))

PyTorch output for 'bear': tensor([0.80, 0.87, 0.67, 1.00])
ours == PyTorch: True


- Each head gives $4 \times 2$, Concat gives $4 \times 4$ and $W_O$ keeps $4 \times 4$: the shapes of the picture.
- For "bear", the two head outputs of 2.1 are glued and mixed by $W_O$ into one vector of $d_{\text{model}} = 4$ numbers.
- The built-in version gives the same numbers. It keeps the $W_Q$ of all heads in one matrix and splits it into heads inside.

#### 2.4 Each head needs its own matrices

Suppose both heads use the same $W_Q$, $W_K$ and $W_V$, those of head 1.

In [5]:
# Two heads with exactly the same matrices (those of head 1)
same = [attention(X, W_Q[0], W_K[0], W_V[0]) for _ in range(2)]
concat_same = torch.cat(same, dim=-1)

print("the two head outputs are equal:", torch.allclose(same[0], same[1]))
print("Concat for 'bear':", concat_same[3])

# W_O on the Concat ...
Z_same = concat_same @ W_O
# ... gives the same as ONE head, with the two halves of W_O added up
Z_one = same[0] @ (W_O[:2] + W_O[2:])
print("two identical heads == one head with a merged W_O:", torch.allclose(Z_same, Z_one, atol=1e-6))

the two head outputs are equal: True
Concat for 'bear': tensor([0.21, 0.40, 0.21, 0.40])
two identical heads == one head with a merged W_O: True


- Both heads give the same output, so Concat holds two copies of the same numbers.
- $W_O$ then just adds the two copies, and one head with a merged $W_O$ does the same. The second head brings no new information.
- This is why the matrices of the heads start from different random numbers. Each head then learns its own way of looking.

## 3. Cross-attention

The lecture example is a translation:

> English (input): A cute teddy bear is reading.
>
> French (output): Un ours en peluche mignon lit.

- The **encoder** reads the English sentence and gives one vector per English token.
- The **decoder** writes the French sentence, one token at a time. To write "ours", it has to look at "bear" in the English sentence.
- Self-attention only gives the decoder the French tokens written so far.
- **Cross-attention** (the lecture calls it encoder-decoder attention) lets each decoder token look at the tokens of the encoder.

#### 3.1 Where Q, K and V come from

We have written the first French words and we are about to write "ours" (bear). Which English word does it translate? "ours" needs information from the English sentence.

The idea is attention again: "ours" looks at the English tokens and takes a weighted mix of them. This is **cross-attention**. Self-attention looks inside one sentence, and cross-attention looks from one sentence into another.

| | Self-attention (notebook 03) | Cross-attention |
|---|---|---|
| Who asks | a token of the sentence | a French token, from the decoder |
| Who is looked at | the tokens of the same sentence | the English tokens, from the encoder |
| Example | "bear" looks at A, cute, teddy, bear | "ours" looks at A, cute, teddy, bear |
| Weight table | one row and one column per token ($4 \times 4$) | one row per French token, one column per English token |

The three roles of notebook 03 follow from this:

- The one who asks makes the **query**. "ours" asks: which English word do I translate? The query is made from the decoder state of "ours" (the vector the decoder has built so far for this word): $Q = (\text{decoder state})\, W_Q$.
- Each English token that is looked at gives a **key** (its label) and a **value** (its content). They are made from the encoder output (the table of English token vectors, one row each): $K = (\text{encoder output})\, W_K$ and $V = (\text{encoder output})\, W_V$.
- The formula stays the same: $\operatorname{softmax}\!\left(Q K^\top / \sqrt{d_k}\right) V$. Only the source of the inputs changes.

Here is where they come from, with numbers:

![The keys and values come from the encoder output, the query comes from the decoder state](images/cross_attention_sources.svg)

- The encoder reads the English sentence once, and every decoder step uses the same $K$ and $V$.
- The numbers are made up: the English table is the matrix $X$ of notebook 03, the matrices are those of notebook 03, and the vector of "ours" is a made-up row.

#### 3.2 How much does "ours" listen to each English token?

3.1 gave "ours" a query, and every English token a key and a value. Now the same steps as in notebook 03, for "ours" only:

![The query of ours against the four keys gives scores, then scaled scores, then weights, then a mix of the values](images/cross_attention_steps.svg)

- **Scores:** the query of "ours" against the key of each English token. A big score is a good match. "bear" matches best ($1.4$).
- **Scale:** divide by $\sqrt{d_k} = \sqrt{2}$, as in notebook 03.
- **Softmax:** the scores become shares that add up to 1. These shares are the **attention weights**: how much "ours" listens to each English token. "bear" gets $0.47$.
- **Mix:** each English value times its weight, added up: $[0.72,\ 0.11]$. This is the new vector for "ours". It carries information from the English sentence.

#### 3.3 The whole sentence at once

In 3.2, "ours" took the steps alone, with four English tokens. Now every French token takes them, and we use the whole sentence of the lecture:

- English, 6 tokens: "A cute teddy bear is reading".
- French, 7 tokens: "[BOS] Un ours en peluche mignon lit".

We leave out the full stop and [EOS]. $Q$ and $K$ are new made-up numbers, so they differ from 3.2.

![Q times K transposed gives the scores, softmax gives the attention weights, weights times V gives the output, for 7 French tokens and 6 English tokens](images/cross_attention_whole.svg)

Read the picture in three steps:

- **$Q \times K^\top$ gives the scores.** Every French query meets every English key. The table has one row per French token and one column per English token ($7 \times 6$). A big score is a good match.
- **$\div \sqrt{d_k}$, then softmax on each row, gives the attention weights.** Each row adds up to 1. They are computed from $Q$ and $K$. The learned parameters are the matrices $W_Q$, $W_K$ and $W_V$ that make $Q$, $K$ and $V$.
- **Weights $\times V$ gives the output.** Each French token gets a mix of the English values: one new vector per French token ($7 \times 2$).
  For "ours": $0.73 \cdot V_{\text{bear}} + 0.18 \cdot V_{\text{teddy}} + 0.09 \cdot V_{\text{is}} + \dots = [0.92,\ 0.01]$, mostly the value of "bear" $[1.3,\ -0.1]$. It is the information that "ours" took from the English sentence.

What the weights show:

- Each French word looks mostly at the English word it translates: "Un" at "A", "ours" at "bear", "peluche" at "teddy", "mignon" at "cute".
- A French word can look at several English words: "en" splits its attention between "teddy" and "bear", and "lit" between "is" and "reading".
- "[BOS]" is the start token and has no preference, so its attention is spread evenly ($1/6$ each).
- A trained model learns patterns like these. Here $Q$ and $K$ are made up to give them.

Let $T_{\text{dec}}$ be the number of French tokens ($7$) and $T_{\text{enc}}$ the number of English tokens ($6$).

| Piece | Comes from | Shape | Our example |
|---|---|---|---|
| $Q$ | decoder | $T_{\text{dec}} \times d_k$ | $7 \times 2$ |
| $K$ | encoder | $T_{\text{enc}} \times d_k$ | $6 \times 2$ |
| $V$ | encoder | $T_{\text{enc}} \times d_v$ | $6 \times 2$ |
| Scores and attention weights | | $T_{\text{dec}} \times T_{\text{enc}}$ | $7 \times 6$ |
| Output (weights $\times V$) | | $T_{\text{dec}} \times d_v$ | $7 \times 2$ |

- $Q$ and $K$ have the same $d_k$, so every query can be compared with every key.
- $K$ and $V$ have one row per English token.
- The output has one row per French token. The number of English tokens disappears in the product with $V$.

#### 3.4 Check in PyTorch

We compute the $7 \times 6$ example of 3.3 from the same $Q$, $K$ and $V$, and compare with the built-in `F.scaled_dot_product_attention`. It accepts queries and keys of different lengths.

In [6]:
import torch.nn.functional as F

en = ["A", "cute", "teddy", "bear", "is", "reading"]
fr = ["[BOS]", "Un", "ours", "en", "peluche", "mignon", "lit"]

# Made-up numbers of the picture. Q: one row per French token. K and V: one row per English token
Q = 2 * torch.tensor([[0., 0.], [-2., 1.], [2., -1.], [2., 0.], [2., 1.], [0., 2.], [-1., -2.]])
K = torch.tensor([[-1., 0.5], [0., 1.], [1., 0.5], [1., -0.5], [0., -1.], [-1., -0.5]])
V = torch.tensor([[0.3, 0.1], [0.4, 0.2], [-0.2, 0.7], [1.3, -0.1], [0.1, -0.5], [-0.6, 0.4]])

scores = Q @ K.T                                                  # (7, 2) x (2, 6) = (7, 6)
weights = torch.softmax(scores / math.sqrt(K.shape[-1]), dim=-1)   # each row adds up to 1
Z = weights @ V                                                   # (7, 6) x (6, 2) = (7, 2)

print("Q:      ", tuple(Q.shape))
print("K:      ", tuple(K.shape))
print("V:      ", tuple(V.shape))
print("scores: ", tuple(scores.shape))
print("weights:", tuple(weights.shape))
print("output: ", tuple(Z.shape))

i = fr.index("ours")
print("\nscores for 'ours':      ", scores[i])
print("weights for 'ours':     ", weights[i], " (columns:", en, ")")
print("output for 'ours':      ", Z[i])
print("rows of weights add up to 1:", torch.allclose(weights.sum(dim=1), torch.ones(7)))

Q:       (7, 2)
K:       (6, 2)
V:       (6, 2)
scores:  (7, 6)
weights: (7, 6)
output:  (7, 2)

scores for 'ours':       tensor([-5., -2.,  3.,  5.,  2., -3.])
weights for 'ours':      tensor([0.00, 0.01, 0.18, 0.73, 0.09, 0.00])  (columns: ['A', 'cute', 'teddy', 'bear', 'is', 'reading'] )
output for 'ours':       tensor([0.92, 0.01])
rows of weights add up to 1: True


In [7]:
# PyTorch's version (it expects a batch dimension, so we add one with [None])
Z_torch = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
print("ours == PyTorch:", torch.allclose(Z, Z_torch, atol=1e-6))

ours == PyTorch: True


Each French token gets its output from its own query. We change the query of "ours" and see which outputs move.

In [8]:
Q_new = Q.clone()
Q_new[fr.index("ours")] = torch.tensor([-8., 4.])                # "ours" now asks like "Un"
Z_new = F.scaled_dot_product_attention(Q_new[None], K[None], V[None])[0]

unchanged = torch.isclose(Z, Z_new, atol=1e-6).all(dim=1)
for name, same in zip(fr, unchanged):
    print(f"{name:8s} output unchanged: {bool(same)}")

[BOS]    output unchanged: True
Un       output unchanged: True
ours     output unchanged: False
en       output unchanged: True
peluche  output unchanged: True
mignon   output unchanged: True
lit      output unchanged: True


- The shapes are those of 3.3: $Q$ is $7 \times 2$, $K$ and $V$ are $6 \times 2$, the weights are $7 \times 6$ and the output is $7 \times 2$.
- The weights of "ours" put most of the attention on "bear" ($0.73$), as in the picture.
- Changing the query of "ours" changes only the output of "ours". The English side ($K$ and $V$) stays the same for every French token.